In [ ]:
#!/usr/bin/env python3
"""
Stitch multiple GeoTIFF files from Google Cloud Storage into a single mosaic.

This script:
1. Downloads GeoTIFF files from a source GCS bucket
2. Merges them into a single mosaic using rasterio
3. Uploads the result to a destination GCS bucket

Requirements:
    pip install google-cloud-storage rasterio numpy

Usage:
    python stitch_geotiffs.py \
        --source-bucket my-source-bucket \
        --source-prefix path/to/geotiffs/ \
        --dest-bucket my-dest-bucket \
        --dest-path output/mosaic.tif

Authentication:
    Set GOOGLE_APPLICATION_CREDENTIALS environment variable to your service account key,
    or run `gcloud auth application-default login` for local development.
"""

import argparse
import os
import tempfile
from pathlib import Path

import numpy as np
import rasterio
from rasterio.merge import merge
from google.cloud import storage


In [ ]:


def list_geotiffs(bucket_name: str, prefix: str = "") -> list[str]:
    """List all GeoTIFF files in a GCS bucket with given prefix."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    
    blobs = bucket.list_blobs(prefix=prefix)
    geotiff_extensions = ('.tif', '.tiff', '.geotiff')
    
    return [
        blob.name for blob in blobs 
        if blob.name.lower().endswith(geotiff_extensions)
    ]


def download_geotiffs(
    bucket_name: str, 
    blob_names: list[str], 
    local_dir: str
) -> list[Path]:
    """Download GeoTIFF files from GCS to local directory."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    
    local_paths = []
    for blob_name in blob_names:
        blob = bucket.blob(blob_name)
        local_filename = Path(blob_name).name
        local_path = Path(local_dir) / local_filename
        
        print(f"Downloading: {blob_name}")
        blob.download_to_filename(str(local_path))
        local_paths.append(local_path)
    
    return local_paths


def stitch_geotiffs(
    input_paths: list[Path], 
    output_path: Path,
    nodata: float | None = None,
    resampling: str = "nearest"
) -> None:
    """
    Merge multiple GeoTIFF files into a single mosaic.
    
    Args:
        input_paths: List of paths to input GeoTIFF files
        output_path: Path for the output mosaic
        nodata: NoData value for the output (auto-detected if None)
        resampling: Resampling method for overlapping areas
    """
    from rasterio.enums import Resampling
    
    resampling_method = getattr(Resampling, resampling, Resampling.nearest)
    
    # Open all source files
    src_files = [rasterio.open(p) for p in input_paths]
    
    try:
        # Get nodata from first file if not specified
        if nodata is None:
            nodata = src_files[0].nodata
        
        print(f"Merging {len(src_files)} files...")
        mosaic, out_transform = merge(
            src_files, 
            nodata=nodata,
            resampling=resampling_method
        )
        
        # Get metadata from first file and update for mosaic
        out_meta = src_files[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "nodata": nodata,
            "compress": "lzw",  # Add compression for smaller file size
            "tiled": True,      # Use tiled format for better performance
            "blockxsize": 256,
            "blockysize": 256,
        })
        
        print(f"Writing mosaic: {output_path}")
        print(f"  Shape: {mosaic.shape}")
        print(f"  Dtype: {mosaic.dtype}")
        
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic)
            
    finally:
        # Close all source files
        for src in src_files:
            src.close()


def upload_to_gcs(local_path: Path, bucket_name: str, dest_blob_name: str) -> str:
    """Upload a file to GCS and return the gs:// URI."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(dest_blob_name)
    
    print(f"Uploading to: gs://{bucket_name}/{dest_blob_name}")
    blob.upload_from_filename(str(local_path))
    
    return f"gs://{bucket_name}/{dest_blob_name}"



In [ ]:

def main():
    parser = argparse.ArgumentParser(
        description="Stitch GeoTIFFs from GCS into a single mosaic"
    )
    parser.add_argument(
        "--source-bucket", "-s",
        required=True,
        help="Source GCS bucket name"
    )
    parser.add_argument(
        "--source-prefix", "-p",
        default="",
        help="Prefix/folder path in source bucket (e.g., 'data/tiles/')"
    )
    parser.add_argument(
        "--dest-bucket", "-d",
        required=True,
        help="Destination GCS bucket name"
    )
    parser.add_argument(
        "--dest-path", "-o",
        default="mosaic.tif",
        help="Destination blob path (e.g., 'output/mosaic.tif')"
    )
    parser.add_argument(
        "--nodata",
        type=float,
        default=None,
        help="NoData value (auto-detected if not specified)"
    )
    parser.add_argument(
        "--resampling",
        choices=["nearest", "bilinear", "cubic", "average", "max", "min"],
        default="nearest",
        help="Resampling method for overlapping areas"
    )
    parser.add_argument(
        "--keep-temp",
        action="store_true",
        help="Keep temporary files after processing"
    )
    
    args = parser.parse_args()
    
    # List GeoTIFFs in source bucket
    print(f"Scanning gs://{args.source_bucket}/{args.source_prefix}")
    geotiff_blobs = list_geotiffs(args.source_bucket, args.source_prefix)
    
    if not geotiff_blobs:
        print("No GeoTIFF files found!")
        return 1
    
    print(f"Found {len(geotiff_blobs)} GeoTIFF files")
    
    # Create temp directory for processing
    with tempfile.TemporaryDirectory(delete=not args.keep_temp) as tmpdir:
        if args.keep_temp:
            print(f"Temp directory: {tmpdir}")
        
        # Download files
        local_paths = download_geotiffs(
            args.source_bucket, 
            geotiff_blobs, 
            tmpdir
        )
        
        # Stitch together
        output_local = Path(tmpdir) / "mosaic.tif"
        stitch_geotiffs(
            local_paths, 
            output_local,
            nodata=args.nodata,
            resampling=args.resampling
        )
        
        # Upload result
        gcs_uri = upload_to_gcs(
            output_local, 
            args.dest_bucket, 
            args.dest_path
        )
        
        print(f"\nSuccess! Mosaic saved to: {gcs_uri}")
    
    return 0


if __name__ == "__main__":
    exit(main())